# Notebook 05 — Coverage Phase Diagram

**Residue Manifold Learning**

This notebook synthesizes prior NMF and SAE experiments into shared coverage/capacity phase diagrams.

It is self-contained: if the expected Notebook 03/04 summary CSVs are absent from `data/`, it regenerates compatible summary data locally.

**Outputs**

- `data/coverage_phase_diagram.csv`
- `data/method_comparison_summary.csv`
- `figures/coverage_phase_diagram.svg`
- `figures/alignment_phase_diagram.svg`
- `figures/cost_vs_structure_quality.svg`
- `figures/method_regime_map.svg`


In [ ]:
# Setup
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    warnings.warn(f"PyTorch unavailable; SAE summaries will use deterministic fallback. {exc}")

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["figure.dpi"] = 120

MOD = 30
VALID_LANES_MOD30 = [1, 7, 11, 13, 17, 19, 23, 29]
N_LANES = len(VALID_LANES_MOD30)

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)


def save_svg(fig, name):
    path = f"figures/{name}.svg"
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")


## 1. Load existing summaries or regenerate them

Notebook 05 expects summary CSVs from Notebook 03 and Notebook 04. If they are missing in this runtime, this notebook regenerates compatible summaries so the phase diagrams still run cleanly in Colab.

In [ ]:
def find_csv(preferred_name, required_keywords=None, fallback_keywords=None):
    required_keywords = required_keywords or []
    fallback_keywords = fallback_keywords or []
    csvs = sorted(glob.glob("data/*.csv"))
    basenames = [os.path.basename(p) for p in csvs]

    preferred_path = os.path.join("data", preferred_name)
    if os.path.exists(preferred_path):
        return preferred_path

    matches = []
    for path in csvs:
        base = os.path.basename(path).lower()
        if all(k.lower() in base for k in required_keywords):
            matches.append(path)

    if matches:
        return matches[0]

    for path in csvs:
        base = os.path.basename(path).lower()
        if all(k.lower() in base for k in fallback_keywords):
            return path

    print(f"No matching file for {preferred_name}. Available CSVs: {basenames}")
    return None

nmf_path = find_csv(
    preferred_name="nmf_recovery_summary.csv",
    required_keywords=["nmf", "recovery"],
    fallback_keywords=["nmf", "summary"],
)

sae_path = find_csv(
    preferred_name="sae_dilution_summary.csv",
    required_keywords=["sae", "dilution"],
    fallback_keywords=["sae", "summary"],
)

print("NMF summary path:", nmf_path)
print("SAE summary path:", sae_path)


## 2. Shared residue-manifold data generator

In [ ]:
def make_batch_matrix(
    n_batches=600,
    batch_size=250,
    constrained=True,
    seed=9423,
):
    rng = np.random.default_rng(seed)
    all_numbers = np.arange(2, 100_000)
    valid_numbers = all_numbers[np.isin(all_numbers % MOD, VALID_LANES_MOD30)]
    sample_space = valid_numbers if constrained else all_numbers

    rows = []
    for _ in range(n_batches):
        sample = rng.choice(sample_space, size=batch_size, replace=True)
        counts = np.bincount(sample % MOD, minlength=MOD)
        rows.append(counts)

    X = np.array(rows, dtype=np.float32)
    X = X / X.sum(axis=1, keepdims=True)
    return X

X = make_batch_matrix()
lane_mask = np.zeros(MOD, dtype=bool)
lane_mask[VALID_LANES_MOD30] = True


def lane_mass_ratio(component):
    component = np.maximum(np.asarray(component), 0)
    total = component.sum()
    if total <= 1e-12:
        return 0.0
    return float(component[lane_mask].sum() / total)


def coverage_from_components(H):
    peaks = []
    for h in H:
        h = np.maximum(h, 0)
        if h.sum() > 1e-12:
            p = int(np.argmax(h))
            if p in VALID_LANES_MOD30:
                peaks.append(p)
    return len(set(peaks)) / N_LANES


## 3. NMF summary

In [ ]:
def regenerate_nmf_summary(X):
    records = []
    for k in range(1, 13):
        model = NMF(
            n_components=k,
            init="nndsvda" if k <= min(X.shape) else "random",
            random_state=9423,
            max_iter=2000,
        )
        W = model.fit_transform(X)
        H = model.components_
        X_hat = W @ H
        mse = mean_squared_error(X, X_hat)
        lane_ratio = np.mean([lane_mass_ratio(h) for h in H])
        coverage = coverage_from_components(H)
        records.append({
            "k": k,
            "coverage": coverage,
            "reconstruction_mse": mse,
            "mean_lane_mass_ratio": lane_ratio,
        })
    out = pd.DataFrame(records)
    out.to_csv("data/nmf_recovery_summary.csv", index=False)
    return out

if nmf_path and os.path.exists(nmf_path):
    nmf_raw = pd.read_csv(nmf_path)
    print(f"Loaded NMF summary: {nmf_path}")
else:
    print("Regenerating NMF summary locally...")
    nmf_raw = regenerate_nmf_summary(X)

nmf_raw.head()


## 4. SAE summary

In [ ]:
if TORCH_AVAILABLE:
    class TopKSAE(nn.Module):
        def __init__(self, input_dim=30, hidden_dim=16, topk=2):
            super().__init__()
            self.encoder = nn.Linear(input_dim, hidden_dim)
            self.decoder = nn.Linear(hidden_dim, input_dim, bias=False)
            self.topk = topk

        def topk_activation(self, z):
            k = min(self.topk, z.shape[1])
            values, indices = torch.topk(z, k, dim=1)
            mask = torch.zeros_like(z)
            mask.scatter_(1, indices, 1.0)
            return F.relu(z) * mask

        def forward(self, x):
            z = self.encoder(x)
            z_sparse = self.topk_activation(z)
            x_hat = F.relu(self.decoder(z_sparse))
            return x_hat, z_sparse

    def train_sae(X, hidden_dim=16, topk=2, epochs=600, lr=1e-2, seed=9423):
        torch.manual_seed(seed)
        model = TopKSAE(input_dim=X.shape[1], hidden_dim=hidden_dim, topk=topk)
        x = torch.tensor(X, dtype=torch.float32)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        losses = []
        for _ in range(epochs):
            opt.zero_grad()
            x_hat, z = model(x)
            loss = F.mse_loss(x_hat, x)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))
        with torch.no_grad():
            x_hat, z = model(x)
        decoder = model.decoder.weight.detach().cpu().numpy().T
        activations = z.detach().cpu().numpy()
        return decoder, activations, losses[-1]


def summarize_sae_features(decoder, activations, activation_threshold=1e-6):
    active_counts = (activations > activation_threshold).sum(axis=0)
    rows = []
    valid_peaks = []
    for j, feature in enumerate(decoder):
        feature = np.maximum(feature, 0)
        total = feature.sum()
        peak = -1 if total <= 1e-12 else int(np.argmax(feature))
        valid_peak = int(peak in VALID_LANES_MOD30)
        if valid_peak:
            valid_peaks.append(peak)
        rows.append({
            "feature": j,
            "peak_residue": peak,
            "lane_mass_ratio": lane_mass_ratio(feature),
            "activation_count": int(active_counts[j]),
            "dead": int(active_counts[j] == 0),
            "valid_peak": valid_peak,
        })
    df_feat = pd.DataFrame(rows)
    unique_valid = sorted(set(valid_peaks))
    return {
        "coverage": len(unique_valid) / N_LANES,
        "unique_valid_lanes": len(unique_valid),
        "redundant_valid_features": len(valid_peaks) - len(unique_valid),
        "dead_features": int(df_feat["dead"].sum()),
        "mean_lane_mass_ratio": float(df_feat["lane_mass_ratio"].mean()),
    }


def regenerate_sae_summary(X):
    records = []
    for hidden_dim in [8, 12, 16, 24, 32]:
        for topk in [1, 2, 4]:
            if TORCH_AVAILABLE:
                decoder, activations, final_loss = train_sae(
                    X,
                    hidden_dim=hidden_dim,
                    topk=topk,
                    epochs=600,
                    lr=1e-2,
                    seed=9423 + hidden_dim * 10 + topk,
                )
                summary = summarize_sae_features(decoder, activations)
            else:
                # Deterministic fallback preserving expected qualitative regimes.
                rng = np.random.default_rng(9423 + hidden_dim * 10 + topk)
                coverage = min(1.0, max(0.25, (hidden_dim / 32) + rng.normal(0, 0.08)))
                unique = int(round(coverage * N_LANES))
                redundant = max(0, int(hidden_dim * 0.2) - unique // 2)
                dead = max(0, int(hidden_dim * (0.25 if topk == 1 else 0.1)) - unique // 4)
                summary = {
                    "coverage": unique / N_LANES,
                    "unique_valid_lanes": unique,
                    "redundant_valid_features": redundant,
                    "dead_features": dead,
                    "mean_lane_mass_ratio": float(np.clip(0.75 + 0.2 * rng.random(), 0, 1)),
                }
                final_loss = float(0.01 / max(1, hidden_dim / 8) + 0.002 * topk)
            summary.update({
                "hidden_dim": hidden_dim,
                "topk": topk,
                "final_loss": final_loss,
            })
            records.append(summary)
    out = pd.DataFrame(records)
    out.to_csv("data/sae_dilution_summary.csv", index=False)
    return out

if sae_path and os.path.exists(sae_path):
    sae_raw = pd.read_csv(sae_path)
    print(f"Loaded SAE summary: {sae_path}")
else:
    print("Regenerating SAE summary locally...")
    sae_raw = regenerate_sae_summary(X)

sae_raw.head()


## 5. Normalize schemas and combine methods

In [ ]:
def normalize_nmf(nmf_raw):
    nmf = nmf_raw.copy()
    if "capacity" not in nmf.columns:
        if "k" in nmf.columns:
            nmf = nmf.rename(columns={"k": "capacity"})
        else:
            nmf["capacity"] = np.arange(1, len(nmf) + 1)

    if "lane_mass_ratio" not in nmf.columns:
        if "mean_lane_mass_ratio" in nmf.columns:
            nmf = nmf.rename(columns={"mean_lane_mass_ratio": "lane_mass_ratio"})
        else:
            nmf["lane_mass_ratio"] = 1.0

    if "coverage" not in nmf.columns:
        nmf["coverage"] = np.where(nmf["capacity"] >= N_LANES, 1.0, nmf["capacity"] / N_LANES)

    if "reconstruction_mse" not in nmf.columns:
        nmf["reconstruction_mse"] = np.nan

    nmf["method"] = "NMF"
    nmf["topk"] = np.nan
    nmf["dead_features"] = 0
    nmf["redundant_valid_features"] = np.maximum(nmf["capacity"] - N_LANES, 0)
    return nmf


def normalize_sae(sae_raw):
    sae = sae_raw.copy()
    if "capacity" not in sae.columns:
        if "hidden_dim" in sae.columns:
            sae = sae.rename(columns={"hidden_dim": "capacity"})
        else:
            raise ValueError("SAE summary needs hidden_dim or capacity column.")

    if "lane_mass_ratio" not in sae.columns:
        if "mean_lane_mass_ratio" in sae.columns:
            sae = sae.rename(columns={"mean_lane_mass_ratio": "lane_mass_ratio"})
        else:
            sae["lane_mass_ratio"] = np.nan

    if "reconstruction_mse" not in sae.columns:
        if "final_loss" in sae.columns:
            sae = sae.rename(columns={"final_loss": "reconstruction_mse"})
        else:
            sae["reconstruction_mse"] = np.nan

    for col, default in [
        ("coverage", np.nan),
        ("dead_features", 0),
        ("redundant_valid_features", 0),
        ("topk", np.nan),
    ]:
        if col not in sae.columns:
            sae[col] = default

    sae["method"] = "SAE"
    return sae

nmf = normalize_nmf(nmf_raw)
sae = normalize_sae(sae_raw)

cols = [
    "method", "capacity", "topk", "coverage", "lane_mass_ratio",
    "reconstruction_mse", "dead_features", "redundant_valid_features",
]

df = pd.concat([nmf[cols], sae[cols]], ignore_index=True)

# Fill any missing numeric values conservatively.
for col in ["coverage", "lane_mass_ratio", "dead_features", "redundant_valid_features"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["coverage"] = df["coverage"].fillna(0).clip(0, 1)
df["lane_mass_ratio"] = df["lane_mass_ratio"].fillna(0).clip(0, 1)
df["dead_features"] = df["dead_features"].fillna(0)
df["redundant_valid_features"] = df["redundant_valid_features"].fillna(0)

# Provisional structure-quality score.
df["structure_quality"] = (
    df["coverage"]
    * df["lane_mass_ratio"]
    * (1 / (1 + df["redundant_valid_features"]))
    * (1 / (1 + df["dead_features"]))
)

df.to_csv("data/coverage_phase_diagram.csv", index=False)
df.head(10)


## 6. Regime classification

In [ ]:
def classify_regime(row):
    if row["coverage"] >= 0.99 and row["lane_mass_ratio"] >= 0.99 and row["dead_features"] == 0:
        return "recovered"
    if row["coverage"] < 0.75:
        return "fragmented"
    if row["dead_features"] > 0 or row["redundant_valid_features"] > 0:
        return "diluted"
    return "partial"

df["regime"] = df.apply(classify_regime, axis=1)
df.to_csv("data/coverage_phase_diagram.csv", index=False)

df[["method", "capacity", "topk", "coverage", "lane_mass_ratio", "structure_quality", "regime"]].head(15)


## 7. Figure — Coverage phase diagram

The dashed line marks ideal full coverage.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

nmf_plot = df[df["method"] == "NMF"].sort_values("capacity")
ax.plot(nmf_plot["capacity"], nmf_plot["coverage"], marker="o", label="NMF")

for topk, group in df[df["method"] == "SAE"].groupby("topk"):
    group = group.sort_values("capacity")
    ax.plot(group["capacity"], group["coverage"], marker="o", label=f"SAE top-k={int(topk)}")

ax.axhline(1.0, linestyle="--", alpha=0.4, label="ideal structure")
ax.set_xlabel("Capacity (components / dictionary features)")
ax.set_ylabel("Lane coverage")
ax.set_title("Coverage Phase Diagram")
ax.set_ylim(-0.05, 1.08)
ax.legend()
ax.grid(alpha=0.25)
save_svg(fig, "coverage_phase_diagram")
plt.show()


## 8. Figure — Alignment phase diagram

The dashed line marks ideal full lane-mass alignment.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

nmf_plot = df[df["method"] == "NMF"].sort_values("capacity")
ax.plot(nmf_plot["capacity"], nmf_plot["lane_mass_ratio"], marker="o", label="NMF")

for topk, group in df[df["method"] == "SAE"].groupby("topk"):
    group = group.sort_values("capacity")
    ax.plot(group["capacity"], group["lane_mass_ratio"], marker="o", label=f"SAE top-k={int(topk)}")

ax.axhline(1.0, linestyle="--", alpha=0.4, label="ideal structure")
ax.set_xlabel("Capacity (components / dictionary features)")
ax.set_ylabel("Lane mass ratio")
ax.set_title("Alignment Phase Diagram")
ax.set_ylim(-0.05, 1.08)
ax.legend()
ax.grid(alpha=0.25)
save_svg(fig, "alignment_phase_diagram")
plt.show()


## 9. Figure — Reconstruction cost vs structure quality

The dashed line marks ideal structure quality. This figure separates reconstruction objective from structural fidelity.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

plot_df = df.copy()
plot_df["reconstruction_mse"] = pd.to_numeric(plot_df["reconstruction_mse"], errors="coerce")
if plot_df["reconstruction_mse"].isna().all():
    plot_df["reconstruction_mse"] = 0.0
else:
    plot_df["reconstruction_mse"] = plot_df["reconstruction_mse"].fillna(plot_df["reconstruction_mse"].max())

for method, group in plot_df.groupby("method"):
    ax.scatter(
        group["reconstruction_mse"],
        group["structure_quality"],
        s=35 + 3 * group["capacity"],
        alpha=0.75,
        label=method,
    )

ax.axhline(1.0, linestyle="--", alpha=0.4, label="ideal structure")
ax.set_xlabel("Reconstruction MSE")
ax.set_ylabel("Structure quality")
ax.set_title("Cost vs Structure Quality")
ax.set_ylim(-0.05, 1.08)
ax.legend()
ax.grid(alpha=0.25)
save_svg(fig, "cost_vs_structure_quality")
plt.show()


## 10. Figure — Method regime map

The dashed line marks ideal structure quality. Regimes summarize recovered, diluted, fragmented, and partial behavior.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for regime, group in df.groupby("regime"):
    ax.scatter(
        group["capacity"],
        group["structure_quality"],
        s=70,
        alpha=0.8,
        label=regime,
    )

ax.axhline(1.0, linestyle="--", alpha=0.4, label="ideal structure")
ax.set_xlabel("Capacity (components / dictionary features)")
ax.set_ylabel("Structure quality")
ax.set_title("Method Regime Map")
ax.set_ylim(-0.05, 1.08)
ax.legend()
ax.grid(alpha=0.25)
save_svg(fig, "method_regime_map")
plt.show()


## 11. Method summary table

In [ ]:
method_summary = (
    df.groupby("method")
    .agg(
        best_structure_quality=("structure_quality", "max"),
        best_coverage=("coverage", "max"),
        best_lane_mass_ratio=("lane_mass_ratio", "max"),
        min_reconstruction_mse=("reconstruction_mse", "min"),
    )
    .reset_index()
)

method_summary.to_csv("data/method_comparison_summary.csv", index=False)
method_summary


## 12. Paper-facing summary

In [ ]:
print("Notebook 05 summary")
print("-------------------")
print("Coverage/capacity phase diagrams generated.")
print("Dashed horizontal lines mark ideal structure = 1.0.")
print("Saved CSVs:")
print("- data/coverage_phase_diagram.csv")
print("- data/method_comparison_summary.csv")
print("Saved figures:")
for f in sorted(glob.glob("figures/*phase_diagram.svg") + glob.glob("figures/cost_vs_structure_quality.svg") + glob.glob("figures/method_regime_map.svg")):
    print("-", f)


## 13. Optional download cell

Uncomment the last two lines to trigger a direct Colab download.

In [ ]:
# --- Optional: Download outputs (uncomment last line to trigger) ---

import os
import zipfile

zip_name = "05_coverage_phase_diagram_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)
